In [1]:
import pandas as pd
import itertools
import networkx as nx
import matplotlib.pyplot as plt
from graph_tool.all import Graph, graph_draw, sfdp_layout
import glob 
import os
import random

In [2]:
def process_coauthorship_graph(authors_df, disease="breast"):

    #authors with more than 5 publications
    authors_with_more_than_5 = (authors_df.groupby("AID").size()[authors_df.groupby("AID").size() > 5].index)

    #unique authors
    filtered_authors_df = authors_df[authors_df["AID"].isin(authors_with_more_than_5)]

    #filter for disease
    disease_authors = filtered_authors_df[filtered_authors_df["Mention"] == disease]

    if disease_authors.empty:
        print(f"No data for {disease} in the dataset. Skipping...")
        return 

    #group by publication
    grouped_authors = disease_authors.groupby("PMID")

    #edges
    coauthorship_edges = []
    for _, group in grouped_authors:
        authors = group[["LastName", "ForeName"]].apply(tuple, axis=1).tolist()
        coauthorship_edges.extend(itertools.combinations(authors, 2))  #create co-author pairs

    #generate network
    coauthorship_graph = nx.Graph()

    #add authors
    for _, group in grouped_authors:
        for _, row in group.iterrows():
            coauthorship_graph.add_node((row["LastName"], row["ForeName"]), gender=row["Gender"])

    #add edges
    coauthorship_graph.add_edges_from(coauthorship_edges)

    #security check
    if len(coauthorship_graph.nodes()) == 0 or len(coauthorship_graph.edges()) == 0:
        print(f"No co-authorship data found for {disease}. Skipping graph creation.")
        return  #skip graph creation for empty data

    #convert to graph-tool for visualization
    g = Graph(directed=False)
    node_mapping = {}

    for node in coauthorship_graph.nodes():
        node_mapping[node] = g.add_vertex()  #create vertex in graph-tool

    for u, v in coauthorship_graph.edges():
        g.add_edge(node_mapping[u], node_mapping[v])

    #assign color
    pcolor = g.new_vertex_property("string")
    gender_colors = {
        "male": "darkorchid",
        "female": "darkorange",
    }

    for node, vertex in node_mapping.items():
        gender = coauthorship_graph.nodes[node].get("gender", "unknown")
        pcolor[vertex] = gender_colors.get(gender, "gray")

    pos = sfdp_layout(g, cooling_step=0.95, epsilon=1e-4, gamma=2)
    output_file = f"{disease}_graph.png"

    graph_draw(
        g,
        pos=pos,
        vertex_fill_color=pcolor,
        edge_color=(1, 1, 1, 0),
        output_size=(3000, 3000),
        output=output_file,
    )

    #save network statistics
    network_stats_file = f"{disease}_stats.txt"
    with open(network_stats_file, "w") as stats_file:
        stats_file.write(f"Number of nodes: {coauthorship_graph.number_of_nodes()}\n")
        stats_file.write(f"Number of edges: {coauthorship_graph.number_of_edges()}\n")
        stats_file.write(f"Density: {nx.density(coauthorship_graph):.4f}\n")
        degrees = dict(coauthorship_graph.degree())
        stats_file.write(f"Max degree: {max(degrees.values())}\n")
        stats_file.write(f"Avg degree: {sum(degrees.values()) / len(degrees):.2f}\n")

In [3]:
df = pd.read_csv('authors_cleaned.csv')

In [4]:
df

,PMID,DescriptorName_UI,LastName,ForeName,AID,ORCID,Journal_JournalIssue_PubDate_MedlineDate,Journal_JournalIssue_PubDate_Year,Publication_Year,Century_Decade,Mention,author_name,Gender
0,11584014,D011471,Fujii,Teruhiko,6757189,NaN,NaN,2002,2002,2000-2010,prostate,Teruhiko Fujii,male
1,11584014,D011471,Wang,Qiming,5191708,NaN,NaN,2002,2002,2000-2010,prostate,Qiming Wang,male
2,11584014,D011471,Blumberg,Peter,3687806,NaN,NaN,2002,2002,2000-2010,prostate,Peter M Blumberg,male
3,11584014,D011471,Ohba,Motoi,8648208,NaN,NaN,2002,2002,2000-2010,prostate,Motoi Ohba,male
4,11584014,D011471,Kuroki,Toshio,2419118,NaN,NaN,2002,2002,2000-2010,prostate,Toshio Kuroki,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1172605,28199152,D001943,Hargreaves,Jonathan,6375005,NaN,NaN,2017,2017,2011-2020,breast,Jonathan Hargreaves,male
1172606,28199152,D001943,Elson,Sarah,9419835,NaN,NaN,2017,2017,2011-2020,breast,Sarah L Elson,female
1172607,28199152,D001943,Joe,Bonnie,1351306,NaN,NaN,2017,2017,2011-2020,breast,Bonnie N Joe,female
1172608,28199152,D001943,Feig,Stephen,4908077,NaN,NaN,2017,2017,2011-2020,breast,Stephen A Feig,male


In [5]:
df['AID'].nunique()

407236

In [6]:
df.columns

Index(['PMID', 'DescriptorName_UI', 'LastName', 'ForeName', 'AID', 'ORCID',
       'Journal_JournalIssue_PubDate_MedlineDate',
       'Journal_JournalIssue_PubDate_Year', 'Publication_Year',
       'Century_Decade', 'Mention', 'author_name', 'Gender'],
      dtype='object')

In [ ]:
def create_bipolar_gender_layout(largest_component_subgraph, disease, decade):
    """
    Create a bipolar layout visualization of the co-authorship network based on gender
    """
    from graph_tool.all import Graph, graph_draw, sfdp_layout
    import numpy as np
    
    # Create a new graph-tool graph
    lc_g = Graph(directed=False)
    lc_node_mapping = {}
    
    # Add vertices
    for node in largest_component_subgraph.nodes():
        lc_node_mapping[node] = lc_g.add_vertex()
    
    # Add edges
    for u, v in largest_component_subgraph.edges():
        lc_g.add_edge(lc_node_mapping[u], lc_node_mapping[v])
    
    # Create color property map
    lc_pcolor = lc_g.new_vertex_property("vector<double>")
    
    # Define colors for genders (RGBA)
    gender_colors = {
        "male": [0.5, 0.0, 0.5, 0.8],    # darkorchid in RGBA
        "female": [1.0, 0.65, 0.0, 0.8],  # darkorange in RGBA
        "unknown": [0.5, 0.5, 0.5, 0.8]   # gray in RGBA
    }
    
    # Create the groups property for the bipolar layout
    groups = lc_g.new_vertex_property("int")
    
    # Assign colors and groups
    for node, vertex in lc_node_mapping.items():
        gender = largest_component_subgraph.nodes[node].get("gender", "unknown")
        lc_pcolor[vertex] = gender_colors.get(gender, gender_colors["unknown"])
        
        # Assign groups for bipolar layout (0 for male, 1 for female, random for unknown)
        if gender == "male":
            groups[vertex] = 0
        elif gender == "female":
            groups[vertex] = 1
        else:
            # For unknown gender, assign randomly to one of the groups
            groups[vertex] = np.random.randint(0, 2)
    
    # Create the bipolar layout using the groups parameter
    lc_pos = sfdp_layout(
        lc_g,
        groups=groups,  # This is the key parameter for bipolar layout
        cooling_step=0.95,
        epsilon=1e-3,
        gamma=5
    )
    
    # Output filename
    bipolar_file = f"{disease}_{decade}_bipolar_gender.png"
    
    # Draw the graph with bipolar layout
    graph_draw(
        lc_g,
        pos=lc_pos,
        vertex_fill_color=lc_pcolor,
        edge_color=(0, 0, 0, 0.2),
        vertex_size=10,
        output_size=(3000, 3000),
        output=os.path.join(output_dir, bipolar_file)
    )
    
    # Also save as PDF
    pdf_file = f"{disease}_{decade}_bipolar_gender.pdf"
    graph_draw(
        lc_g,
        pos=lc_pos,
        vertex_fill_color=lc_pcolor,
        edge_color=(0, 0, 0, 0.2),
        vertex_size=10,
        output_size=(3000, 3000),
        output=os.path.join(output_dir, pdf_file)
    )
    
    print(f"Bipolar gender-based layout saved as {bipolar_file} and {pdf_file}")

In [9]:
def process_coauthorship_graph(authors_df, disease, decade):

    output_dir = "Plot_decades"

    #filter out authors with less than 5 publications overall (all 20 years)
    total_author_counts = authors_df.groupby("AID").size()
    authors_with_at_least_5 = total_author_counts[total_author_counts >= 5].index
    prolific_authors_df = authors_df[authors_df["AID"].isin(authors_with_at_least_5)].copy()

    year_df = prolific_authors_df[prolific_authors_df["Century_Decade"] == decade].copy()

    #filter for disease
    disease_authors = year_df[year_df["Mention"] == disease].copy()

    if disease_authors.empty:
        print(f"No data for {disease} in the dataset. Skipping...")
        return 

    #group by publication
    grouped_authors = disease_authors.groupby("PMID")

    #edges
    coauthorship_edges = []
    for _, group in grouped_authors:
        authors = group[["LastName", "ForeName"]].apply(tuple, axis=1).tolist()
        coauthorship_edges.extend(itertools.combinations(authors, 2))  #create co-author pairs

    #generate NETWORKX GRAPH
    coauthorship_graph = nx.Graph()

    #add authors
    for _, group in grouped_authors:
        for _, row in group.iterrows():
            coauthorship_graph.add_node((row["LastName"], row["ForeName"]), gender=row["Gender"])

    #add edges
    coauthorship_graph.add_edges_from(coauthorship_edges)

    #security check
    if len(coauthorship_graph.nodes()) == 0 or len(coauthorship_graph.edges()) == 0:
        print(f"No co-authorship data found for {disease}. Skipping graph creation.")
        return
    
    #connected components
    connected_components = list(nx.connected_components(coauthorship_graph))
    
    #largest connected component
    if connected_components:
        largest_component = max(connected_components, key=len)
        largest_component_subgraph = coauthorship_graph.subgraph(largest_component).copy()
        
        # Log information about the largest component
        print(f"Full graph has {coauthorship_graph.number_of_nodes()} nodes and {len(connected_components)} components")
        print(f"Largest component has {largest_component_subgraph.number_of_nodes()} nodes " 
              f"({largest_component_subgraph.number_of_nodes()/coauthorship_graph.number_of_nodes()*100:.1f}% of full graph)")
    else:
        print("No connected components found.")
        return
    
    #convert to GRAPHTOOL for visualization
    g = Graph(directed=False)
    node_mapping = {}

    for node in coauthorship_graph.nodes():
        node_mapping[node] = g.add_vertex() 

    for u, v in coauthorship_graph.edges():
        g.add_edge(node_mapping[u], node_mapping[v])

    #assign color
    pcolor = g.new_vertex_property("string")
    in_largest = g.new_vertex_property("bool")
    gender_colors = {
        "male": "darkorchid",
        "female": "darkorange",
    }

    for node, vertex in node_mapping.items():
        gender = coauthorship_graph.nodes[node].get("gender", "unknown")
        pcolor[vertex] = gender_colors.get(gender, "gray")
        in_largest[vertex] = node in largest_component

    pos = sfdp_layout(g, cooling_step=0.95, epsilon=1e-4, gamma=2)
    output_file = os.path.join(output_dir, f"full_network{disease}_{decade}.png")

    graph_draw(
        g,
        pos=pos,
        vertex_fill_color=pcolor,
        edge_color=(0, 0, 0, 0),
        output_size=(3000, 3000),
        output=output_file,
    )

    #LARGEST COMPONENT    
    lc_g = Graph(directed=False)
    lc_node_mapping = {}
    
    for node in largest_component_subgraph.nodes():
        lc_node_mapping[node] = lc_g.add_vertex()
    
    for u, v in largest_component_subgraph.edges():
        lc_g.add_edge(lc_node_mapping[u], lc_node_mapping[v])
    
    lc_pcolor = lc_g.new_vertex_property("string")
    
    for node, vertex in lc_node_mapping.items():
        gender = largest_component_subgraph.nodes[node].get("gender", "unknown")
        lc_pcolor[vertex] = gender_colors.get(gender, "gray")
    
    lc_pos = sfdp_layout(lc_g, cooling_step=0.95, epsilon=1e-3, gamma=5)
    
    ### normal graphtool
    graph_draw(
        lc_g,
        pos=lc_pos,
        vertex_fill_color=lc_pcolor,
        edge_color=(0, 0, 0, 0),  
        vertex_size=10,  
        output_size=(3000, 3000),
        output=os.path.join(output_dir, f"{disease}_{decade}_graphtool_normal_largest_component.png") #as png
    )

    graph_draw(        
        lc_g,
        pos=lc_pos,
        vertex_fill_color=lc_pcolor,
        edge_color=(0, 0, 0, 0),  
        vertex_size=10, 
        output=os.path.join(output_dir, f"{disease}_{decade}graphtool_normal_largest_component.pdf")) #as PDF

    ### bipolar
    #create_bipolar_gender_layout(largest_component_subgraph, disease, decade)
    
    ### NetworkX
    plt.figure(figsize=(14, 12))
    pos_nx = nx.spring_layout(largest_component_subgraph, seed=42)
    
    node_colors = []
    for node in largest_component_subgraph.nodes():
        gender = largest_component_subgraph.nodes[node].get("gender", "unknown")
        node_colors.append(gender_colors.get(gender, "gray"))
    
    nx.draw_networkx_nodes(largest_component_subgraph, pos_nx, node_color=node_colors, alpha=0.8)
    nx.draw_networkx_edges(largest_component_subgraph, pos_nx, alpha=0.2, width=0.5)
    
    diameter = nx.diameter(largest_component_subgraph)
    avg_path_length = nx.average_shortest_path_length(largest_component_subgraph)
    density = nx.density(largest_component_subgraph)
    avg_clustering = nx.average_clustering(largest_component_subgraph)
    
    plt.tight_layout()
    plt.savefig(f"{disease}_{decade}_networkx_largest_component.png", dpi=600)
    plt.close()

    #save network statistics
    network_stats_file = os.path.join(output_dir, f"{disease}_{decade}_stats.txt")
    with open(network_stats_file, "w") as stats_file:
        stats_file.write(f"Number of nodes: {coauthorship_graph.number_of_nodes()}\n")
        stats_file.write(f"Number of edges: {coauthorship_graph.number_of_edges()}\n")
        stats_file.write(f"Density: {nx.density(coauthorship_graph):.4f}\n")
        degrees = dict(coauthorship_graph.degree())
        stats_file.write(f"Max degree: {max(degrees.values())}\n")
        stats_file.write(f"Avg degree: {sum(degrees.values()) / len(degrees):.2f}\n")
        
        # Add largest component statistics
        stats_file.write("\nLargest Connected Component Statistics:\n")
        stats_file.write(f"Number of nodes: {largest_component_subgraph.number_of_nodes()}\n")
        stats_file.write(f"Number of edges: {largest_component_subgraph.number_of_edges()}\n")
        stats_file.write(f"Percentage of total nodes: {largest_component_subgraph.number_of_nodes()/coauthorship_graph.number_of_nodes()*100:.2f}%\n")
        stats_file.write(f"Density: {nx.density(largest_component_subgraph):.4f}\n")
        stats_file.write(f"Diameter: {diameter}\n")
        stats_file.write(f"Average shortest path length: {avg_path_length:.4f}\n")
        stats_file.write(f"Average clustering coefficient: {avg_clustering:.4f}\n")
        
        # Add gender distribution in largest component
        gender_counts = {"male": 0, "female": 0, "unknown": 0}
        for node in largest_component_subgraph.nodes():
            gender = largest_component_subgraph.nodes[node].get("gender", "unknown")
            gender_counts[gender] += 1
        
        stats_file.write("\nGender Distribution in Largest Component:\n")
        for gender, count in gender_counts.items():
            stats_file.write(f"{gender.capitalize()}: {count} ({count/largest_component_subgraph.number_of_nodes()*100:.2f}%)\n")

In [14]:
disease = 'prostate'

decades = df['Century_Decade'].unique()
for decade in decades:
    print(f"Processing {disease} co-authorship network for decade: {decade}")
    process_coauthorship_graph(df, disease=disease, decade=decade)

Processing prostate co-authorship network for decade: 2000-2010
Full graph has 17502 nodes and 1574 components
Largest component has 14696 nodes (84.0% of full graph)
Processing prostate co-authorship network for decade: 2011-2020
Full graph has 22352 nodes and 1761 components
Largest component has 19610 nodes (87.7% of full graph)


In [13]:
disease = 'breast'

decades = df['Century_Decade'].unique()
for decade in decades:
    print(f"Processing {disease} co-authorship network for decade: {decade}")
    process_coauthorship_graph(df, disease=disease, decade=decade)

Processing breast co-authorship network for decade: 2000-2010
Full graph has 27848 nodes and 2485 components
Largest component has 23935 nodes (85.9% of full graph)
Processing breast co-authorship network for decade: 2011-2020
Full graph has 34916 nodes and 2112 components
Largest component has 31964 nodes (91.5% of full graph)
